 # Hybrid search for POC terms: Barriere
 

### 1. all results ordered by distance -> determine cosine distance threshold

In [15]:
query_text_sauberkeit = (
    "Sauberkeit: Erwähnung von sauber, dreckig, schmutzig, hygienisch, unhygienisch, "
    "gepflegt, ungepflegt, rein, unrein, ordentlich, unordentlich, Müll, Gestank, Geruch, "
    "saubere Umgebung, schmutzige Umgebung."
)

query_text_akustik = (
    "Akustik: Erwähnung von laut, leise, Geräuschpegel, Lärm, Stille, ruhige Atmosphäre, "
    "laute Umgebung, hallend, dumpf, klarer Klang, schlechter Klang, Musiklautstärke, "
    "Hintergrundgeräusche."
)



query_text_barrierefreiheit = (
    "Barrierefreiheit: Erwähnung von barrierefrei, Rollstuhlzugang, Aufzug vorhanden, "
    "kein Aufzug, Stufen, Treppen, behindertengerecht, leicht zugänglich, schwer zugänglich, "
    "für Menschen mit Behinderung geeignet, Zugang für Rollstuhlfahrer, barrierearme Umgebung, Rollstuhlfahrer"
)

keywords_barrierefreiheit = ['barrierefreiheit', 'behindertengerecht', 'rollstuhl']



query_text_erreichbarkeit = (
    "Erreichbarkeit: Erwähnung von gut erreichbar, schwer erreichbar, mit öffentlichen "
    "Verkehrsmitteln erreichbar, Parkplatz, Parkmöglichkeiten, Anfahrt, Lage, nahe gelegen, "
    "weit entfernt, Haltestelle in der Nähe, Verkehrsanbindung, Wegbeschreibung."
)


query_text_kaffee = (
    "Kaffee: Erwähnung von Kaffee, Espresso, Cappuccino, Latte Macchiato, Kaffeegeschmack, "
    "Kaffeebohnen, frisch gebrüht, bitter, mild, Kaffeequalität, Kaffeehaus, Barista, "
    "guter Kaffee, schlechter Kaffee, Kaffeetasse, Koffeingehalt."
)


### example 
query_text_preis = (
    "Preis: Erwähnung des Preises, der Kosten, ob etwas teuer oder günstig war oder gratis, "
    "Preis-Leistungs-Verhältnis, ob jemand zu viel bezahlt hat, ob es sich gelohnt hat, "
    "überteuert, preiswert, billig, angemessen, nicht wert, günstig."
)


In [ ]:
import psycopg2
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# TODO: replace query text
vector = model.encode(query_text_sauberkeit)
vector_str = str(vector.tolist())


# TODO: replace keywords
like_clauses = " OR ".join([f"LOWER(aspect) = '{kw}'" for kw in keywords_barrierefreiheit])

sql = f"""
SELECT review_id, aspect, aspect_id, sentiment, confidence, snippet_4,
       embedding_4  <=> '{vector_str}'::vector AS distance
FROM aspect_sentiment_results
WHERE embedding_4 <=> '{vector_str}'::vector < 0.75
OR {like_clauses}
ORDER BY distance

"""

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

with conn.cursor() as cur:
    cur.execute(sql)
    rows = cur.fetchall()
    colnames = [desc[0] for desc in cur.description]

conn.close()

df_thresh = pd.DataFrame(rows, columns=colnames)
df_thresh


In [6]:
df_thresh.shape

(188001, 7)

In [7]:
df_thresh["review_id"].nunique()

130697

In [9]:
df_thresh.to_parquet("/Users/lorenaraichle/Developer/ABSA/PyABSA/results/POC/barriere_teil.parquet")

### which aspects have been classified by keyword vs embedding match in hybrid retrieval

In [10]:
df_thresh['match_type'] = df_thresh.apply(
    lambda row: 'keyword' if any(kw in row['aspect'].lower() for kw in keywords_barrierefreiheit)
                else 'embedding' if row['distance'] < 0.75
                else 'none',
    axis=1
)

# TODO: update here keywords 


### check keyword based matches and extend to aspect_keyword as a match type

In [11]:
df_keyword = df_thresh[df_thresh['match_type'] == 'keyword']
df_keyword.shape


(45, 8)

In [12]:
df_keyword["aspect"].nunique()

17

In [13]:
df_keyword["aspect"].unique()

array(['Rollstuhlplätze', 'Rollstuhlparkplatz', 'Rollstuhl',
       'ein Rollstuhl', 'Rollstuhl toilette', 'Rollstuhl Lift',
       'Rollstuhlzentrum', 'Rollstuhlgängigkeit', 'Rollstuhl Fahrer',
       'Rollstuhl WC', 'Rollstuhl Toiletten', 'Rollstuhltaxi',
       'Barrierefreiheit', 'Rollstuhlauswahl', 'Elektro Rollstuhl',
       'Rollstuhlfahrer', 'rollstuhl'], dtype=object)

### EXPANDED KEYWORD CHECK: prefilter google_maps_reviews that mention any of the predicted aspects (keyword_aspects) 

In [14]:
import psycopg2
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

aspect_map = df_thresh.groupby("review_id")["aspect"].apply(list).to_dict()

# set of predicted aspects you want to search in full google_maps_reviews -> Only use aspects predicted in 'keyword'-matched entries
keyword_aspects = set(df_keyword['aspect'].str.lower().unique())

# Query review texts that contain any of the keyword aspects ===
like_clauses = " OR ".join([f"LOWER(review) LIKE '%{kw}%'" for kw in keyword_aspects])

review_sql = f"""
    SELECT id, review
    FROM google_maps_reviews
    WHERE {like_clauses}
"""

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

with conn.cursor() as cur:
    cur.execute(review_sql)
    review_rows = cur.fetchall()

conn.close()



DatabaseError: could not receive data from server: Operation timed out
SSL SYSCALL error: Operation timed out


In [ ]:
matches = []

for review_id, text in review_rows:
    text_lower = text.lower()
    matched_keywords = [kw for kw in keyword_aspects if kw in text_lower]
    
    if matched_keywords:
        # aspects predicted by ABSA model for this review_id (from df_thresh / aspect_map)
        predicted_aspects = [a.lower() for a in aspect_map.get(review_id, [])]
        

        matches.append({
            "review_id": review_id,
            "matched_keywords": matched_keywords,
            "review_text": text_lower,
            "predicted_aspects": predicted_aspects,
        })

df_matches = pd.DataFrame(matches)

# === Check which reviews were already in df_thresh ===
df_matches["in_df_thresh"] = df_matches["review_id"].isin(df_thresh["review_id"])

# === Extract new reviews not found in initial hybrid search ===
df_new_matches = df_matches[~df_matches["in_df_thresh"]]

# merging back
df_new = df_new_matches.copy()
df_new["aspect"] = df_new["matched_keywords"].apply(lambda kws: ", ".join(kws) if kws else None)
df_new["aspect_id"] = df_new["review_id"].astype(str) + "_kw"
df_new["sentiment"] = None
df_new["confidence"] = None
df_new["snippet_4"] = None
df_new["distance"] = None
df_new["match_type"] = "aspect_keyword"

common_cols = df_thresh.columns.intersection(df_new.columns)

df_new_aligned = df_new[common_cols]
df_thresh_aligned = df_thresh[common_cols]

df_combined = pd.concat([df_thresh_aligned, df_new_aligned], ignore_index=True)




In [ ]:
# How many entries per match_type
print(df_combined["match_type"].value_counts())

# How many unique review_ids in combined result
print(f"Unique review_ids: {df_combined['review_id'].nunique()}")

# OLD:  embedding & experimenting with distance filter threshold (from combined_df exlcuding stepwise)

In [196]:
df_embd = df_thresh[df_thresh['match_type'] == 'embedding']
df_embd.shape

(2936, 8)

In [198]:
df_embd["distance"].max()

np.float64(0.8399968425294216)

In [239]:
df_embd_0_8= df_combined[
    (df_combined["match_type"] == "embedding") & 
    (df_combined["distance"] < 0.80)
]
df_embd_0_8

,review_id,aspect,aspect_id,sentiment,confidence,snippet_4,distance,match_type
2,6425011,Speisekarte,6425011_2,Positive,0.9979,herrlichem Weitblick . Grosse Speisekarte mit ...,0.778467,embedding
6,6425035,Aussicht,6425035_3,Positive,0.9986,"Essen , und die Aussicht unbezahlbar .",0.721393,embedding
7,6425044,Ort,6425044_1,Positive,0.9887,Ort & Bedienung sehr toll,0.794187,embedding
8,6425044,Bedienung,6425044_2,Positive,0.9482,Ort & Bedienung sehr toll & sehr,0.777370,embedding
9,6425047,Besuch,6425047_1,Positive,0.9987,welcher Witterung ! Ein Besuch lohnt sich auf ...,0.799081,embedding
...,...,...,...,...,...,...,...,...
3241,6444171,price,6444171_3,Positive,0.8081,bar with games . Value/price worked for us .,0.628544,embedding
3243,6444191,Essen,6444191_1,Positive,0.9981,Essen Gastfreundschaft und Service super,0.796082,embedding
3246,6444193,Zimmer,6444193_1,Neutral,0.8782,Die Zimmer entsprechen den Preisen .,0.650739,embedding
3247,6444193,Platz,6444193_2,Positive,0.7468,Preisen . Grosszüg vom Platz . Etwas weiche Be...,0.632977,embedding
